In [ ]:
using LinearAlgebra
using BoundaryValueDiffEq
using SparseArrays
using Plots
using BenchmarkTools
using ModelingToolkit
using DifferentialEquations

In [ ]:
@parameters gamma, gamma_3, c, omega, x, t
@variables A(..), B(..)
r1 = @rule cos(~x)^3 => 0.75 * cos(~x) + 0.25 * cos(3 * ~x)
r2 = @rule sin(~x)^3 => 0.75 * sin(~x) - 0.25 * sin(3 * ~x)
r3 = @rule cos(~x)^2 => 1 - sin(~x)^2
r4 = @rule sin(~x)^2 => 1 - cos(~x)^2

u = A(x) * sin(omega*t) + B(x) * cos(omega*t)
Dx = Differential(x)
Dt = Differential(t)
y = Dt(Dt(u)) - c*c*Dx(Dx(u)) + gamma*Dt(u) # + gamma_3*Dt(u)*Dt(u)*Dt(u)
y_exp = expand_derivatives(y)

y_exp = simplify(expand(y_exp), RuleSet([r1, r2, r3, r4]))
y_exp = expand(y_exp)
y_exp = simplify(y_exp, RuleSet([r1, r2, r3, r4]))

sin_coeff = Symbolics.coeff(y_exp, sin(omega*t))
cos_coeff = Symbolics.coeff(y_exp, cos(omega*t))

sin_sol = Symbolics.solve_for(sin_coeff ~ 0, Differential(x)(Differential(x)(A(x))))
println("Sin:")
println("Sin coeff: ", sin_coeff)
println("Axx = ", sin_sol)

fA = Symbolics.build_function(sin_sol, A(x), B(x), gamma, omega, c; expression=Val{false})

cos_sol = Symbolics.solve_for(cos_coeff ~ 0, Differential(x)(Differential(x)(B(x))))
println("Cos:")
println("Cos coeff: ", cos_coeff)
println("Bxx = ", cos_sol)

fB = Symbolics.build_function(cos_sol, A(x), B(x), gamma, omega, c; expression=Val{false})

In [ ]:
# set source function
src(x) = 1000. # obtain u(x) > 1 for [u(x)}^3 to be large in size to be important in the non-linearity

# set right-hand side
function rhs!(du, u, p, x)
    c, gamma, gamma3, omd = p
    A = u[1]; dA = u[2]; B = u[3]; dB = u[4];
    du[1] = dA
    du[2] = fA(A, B, gamma, omd, c)
    du[3] = dB
    du[4] = fB(A, B, gamma, omd, c) - src(x)/c^2
end

# set boundary conditions
# boundary conditions on the left and right of computational domain
# requires explaining syntax
function bc(residual, u, p, x)
    Aleft = 0; Aright=0;
    Bleft = 0; Bright=0;
    residual[1] = u[1][1] - Aleft
    residual[2] = u[end][1] - Aright
    residual[3] = u[1][3] - Bleft
    residual[4] = u[end][3] - Bright
end

In [ ]:
# set constants
c = 1.
gamma = 10
gamma3 = 0
omd = 10
p = [c, gamma, gamma3, omd]

omdstep = 1.
omdvec  = 0.:omdstep:10. |> collect

# set domain size
xspan = (0.,1.)

# set initial guess
start = [0., 0., 0., 0.]

# store parameter value in p
p = [c, gamma, gamma3, omd]

# define and solve the problem
bvp = BVProblem(rhs!, bc, start, xspan,p)
sol = solve(bvp);

In [ ]:
# plot u(x) and du/dx(x)
myA  = [u[1] for u in sol.u]
mydA = [u[2] for u in sol.u]
myB  = [u[3] for u in sol.u]
mydB = [u[4] for u in sol.u]
p1 = plot(sol.t, myA, xlabel="x", ylabel = "A(x)")
p2 = plot(sol.t, mydA, xlabel="x", ylabel = "dA(x)")
p3 = plot(sol.t, myB, xlabel="x", ylabel = "B(x)")
p4 = plot(sol.t, mydB, xlabel="x", ylabel = "dB(x)")
plot(p1,p2,p3,p4,layout=(2,2))

In [ ]:
Nt = 1
dt = 0.01
A = [x[1] for x in sol.u] 
B = [x[3] for x in sol.u] 
animation = @animate for k in 1:(Nt/dt)
    new_u = A * sin(omd * (k * dt)) .+ B * cos(omd * (k * dt))
    plot(sol.t, new_u,
         title="t = $(k * dt)", xlabel="x", ylabel="u(x,t)", ylim = (-20, 20))
end

gif(animation, "Example_harmonic.gif", fps = 10)